# TP 01 · Une baseline sans fuite
120 minutes. Livrable : protocole écrit, pipeline, comparaison à une baseline et interprétation de la dispersion.
On conserve le test final fermé. L'AP est l'average precision, pas l'aire trapézoïdale de la courbe PR.
Sources : https://scikit-learn.org/stable/common_pitfalls.html et https://scikit-learn.org/stable/modules/cross_validation.html

Ce bloc met en place une évaluation propre de deux modèles de classification sur le jeu de données *bank* : une baseline naïve et une régression logistique. Il garde un jeu de test final intact et estime les performances uniquement par validation croisée sur les données de développement.

## Imports et chemin du projet

```python
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "mlcourse.py").exists())
sys.path.insert(0, str(ROOT))
```

- `Path.cwd()` désigne le dossier courant du notebook.
- `Path.cwd().parents` fournit les dossiers parents successifs.
- La compréhension cherche le premier dossier contenant `mlcourse.py`, qui devient `ROOT`.
- `sys.path.insert(0, str(ROOT))` ajoute ce dossier en priorité aux chemins où Python cherche les modules : on peut alors écrire `from mlcourse import ...`.

```python
import numpy as np
import pandas as pd
...
from mlcourse import split_bank, bank_pipeline, cv3, SEED
```

- `pandas` (`pd`) servira à afficher les résultats sous forme de tableau.
- `DummyClassifier`, `LogisticRegression` et `cross_validate` viennent de scikit-learn.
- `split_bank`, `bank_pipeline` et `cv3` sont des utilitaires fournis par le cours.
- Dans cette cellule, `np` et `SEED` sont importés mais ne sont pas utilisés directement ; ils peuvent être supprimés sans modifier le résultat.

## Séparer développement et test

```python
X_dev, X_test, y_dev, y_test = split_bank()
assert set(X_dev.index).isdisjoint(X_test.index)
```

`split_bank()` produit quatre objets : les variables explicatives (`X`) et la cible (`y`), chacun séparé entre données de développement (`dev`) et données de test final (`test`).

L’instruction `assert` vérifie que les indices de `X_dev` et de `X_test` n’ont aucun élément commun. C’est une protection contre une fuite de données : une même observation ne doit jamais servir à la fois pour entraîner/valider et pour tester.

Point important : `X_test` et `y_test` ne sont volontairement pas utilisés ensuite. Ils restent « fermés » jusqu’au choix final du modèle et évitent d’adapter les décisions au jeu de test.

## Deux modèles comparés

```python
models = {
    "Prévalence": bank_pipeline(DummyClassifier(strategy="prior")),
    "Logistique": bank_pipeline(
        LogisticRegression(C=1, max_iter=2000)
    ),
}
```

Le dictionnaire `models` associe un nom lisible à chaque modèle complet. Chaque classifieur est placé dans `bank_pipeline(...)`, ce qui encapsule les prétraitements nécessaires avec le modèle. Ainsi, les transformations sont apprises séparément dans chaque pli de validation croisée, plutôt que sur toutes les données à l’avance.

| Modèle | Rôle |
|---|---|
| **Prévalence** | `DummyClassifier(strategy="prior")` ne cherche aucun signal dans les variables. Il prédit une probabilité égale à la proportion globale de la classe positive : c’est la baseline minimale à battre. |
| **Logistique** | `LogisticRegression` apprend une relation entre les variables d’entrée et la probabilité de la classe cible. `C=1` règle la force de régularisation, et `max_iter=2000` laisse davantage d’itérations pour assurer la convergence. |

## Validation croisée

```python
rows = []

for name, model in models.items():
    scores = cross_validate(
        model, X_dev, y_dev,
        cv=cv3(),
        scoring={"AP": "average_precision", "ROC": "roc_auc"},
        n_jobs=1
    )
```

La boucle évalue successivement chaque modèle sur **les mêmes données de développement**. `cv=cv3()` définit une validation croisée à trois plis : à chaque tour, le modèle est entraîné sur une partie des données puis évalué sur la partie restante.

`cross_validate` renvoie un score par pli et par métrique. `n_jobs=1` impose une exécution sur un seul processus, ce qui est souvent plus simple et plus stable dans un notebook.

Les deux métriques demandées sont :

- **AP** (`average_precision`) : mesure la qualité du classement des exemples positifs, particulièrement informative lorsque la classe positive est rare.
- **ROC AUC** (`roc_auc`) : mesure la capacité du modèle à attribuer en moyenne un score plus élevé à un positif qu’à un négatif ; une valeur de 0,5 correspond à un classement aléatoire.

## Agréger les résultats

```python
rows.append({
    "modèle": name,
    "AP moyenne": scores["test_AP"].mean(),
    "AP écart-type": scores["test_AP"].std(),
    "ROC moyenne": scores["test_ROC"].mean()
})
```

Pour chaque modèle, le code conserve :

- la moyenne des scores AP sur les trois plis ;
- l’écart-type de l’AP, qui indique la variabilité selon le pli ;
- la moyenne de l’AUC-ROC.

Un bon score moyen ne suffit donc pas : un écart-type élevé peut révéler une performance instable ou sensible à la manière dont les données sont découpées.

## Affichage et lecture

```python
print(pd.DataFrame(rows).to_string(index=False))
```

Cette instruction transforme la liste `rows` en tableau Pandas puis l’affiche sans colonne d’index technique.

Dans l’exécution affichée, la baseline « Prévalence » atteint une AP d’environ 0,110 et une ROC AUC de 0,5, tandis que la régression logistique atteint une AP d’environ 0,432 et une ROC AUC d’environ 0,777. La régression logistique fait donc nettement mieux que le modèle qui ignore les variables, mais cette conclusion reste une estimation obtenue sur les données de développement ; le test final doit rester réservé à l’évaluation finale.

In [2]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mlcourse.py").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from mlcourse import split_bank, bank_pipeline, cv3, SEED
X_dev, X_test, y_dev, y_test = split_bank()
assert set(X_dev.index).isdisjoint(X_test.index)
models = {
    "Prévalence": bank_pipeline(DummyClassifier(strategy="prior")),
    "Logistique": bank_pipeline(LogisticRegression(C=1, max_iter=2000)),
}
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_dev, y_dev, cv=cv3(),
        scoring={"AP": "average_precision", "ROC": "roc_auc"}, n_jobs=1)
    rows.append({"modèle": name, "AP moyenne": scores["test_AP"].mean(),
                 "AP écart-type": scores["test_AP"].std(),
                 "ROC moyenne": scores["test_ROC"].mean()})
print(pd.DataFrame(rows).to_string(index=False))

    modèle  AP moyenne  AP écart-type  ROC moyenne
Prévalence    0.109560       0.000382     0.500000
Logistique    0.432098       0.032880     0.776519


## Exercice 1 (25 min). Comparer C = 0.01, 0.1, 1 et 10 sur les mêmes plis. Expliquer le sens de C.

### Details de l'exercice

L’objectif est de comparer quatre valeurs de `C (0.01, 0.1, 1, 10)` avec exactement le même protocole : pipeline identique, données `X_dev/y_dev`, validation croisée à trois plis et métrique AP


### Que Signinfie C

Dans LogisticRegression, `C` est l’inverse de la force de régularisation. Autrement dit :
| Valeur de `C`          | Régularisation | Effet attendu                                                                                                  |
| -------------------- | -------------- | -------------------------------------------------------------------------------------------------------------- |
| Petite, par ex. `0.01` | Forte          | Le modèle est contraint, ses coefficients tendent à être plus petits et il risque de sous-apprendre. localhost |
| Grande, par ex. `10`   | Faible         | Le modèle est plus libre de s’ajuster aux données et peut davantage surapprendre. localhost                    |

In [3]:
comparison = []
for C in [0.01, 0.1, 1, 10]:
    model = bank_pipeline(LogisticRegression(C=C, max_iter=2000))
    result = cross_validate(model, X_dev, y_dev, cv=cv3(), scoring="average_precision", n_jobs=1)
    comparison.append((C, result["test_score"].mean(), result["test_score"].std()))
print(pd.DataFrame(comparison, columns=["C", "AP", "écart-type"]))
print("Un C petit renforce la régularisation. La dispersion entre plis n'est pas un intervalle de confiance indépendant.")

       C        AP  écart-type
0   0.01  0.426168    0.005544
1   0.10  0.435013    0.018327
2   1.00  0.432098    0.032880
3  10.00  0.428780    0.038655
Un C petit renforce la régularisation. La dispersion entre plis n'est pas un intervalle de confiance indépendant.


## Exercice 2 (20 min). Faire cinq permutations de la cible et vérifier si un signal subsiste. Ne pas interpréter ce mini-test comme un test statistique définitif.

## Objectif
répondre à la question : « Le modèle exploite-t-il réellement une relation entre les variables `X` et la cible `y` ? »

## Principe

`rng.permutation(y_dev)` mélange aléatoirement les étiquettes de la cible. Par exemple, une ligne qui était positive peut devenir négative, et inversement.

Les valeurs de `X_dev` restent inchangées, mais leur lien avec la cible est volontairement détruit. Il ne doit donc plus y avoir de signal prédictif exploitable.

`SEED` fixe le générateur aléatoire : si les étudiants relancent le notebook dans les mêmes conditions, ils obtiennent les mêmes permutations. Cela améliore la reproductibilité.

## Pourquoi cinq fois ?
Une permutation produit un résultat aléatoire. Répéter l’opération cinq fois donne une petite idée de la variabilité des scores lorsqu’il n’existe aucun lien réel entre les entrées et la cible.

In [4]:
rng = np.random.default_rng(SEED)
null_scores = []
for i in range(5):
    yp = pd.Series(rng.permutation(y_dev), index=y_dev.index)
    result = cross_validate(models["Logistique"], X_dev, yp, cv=cv3(), scoring="average_precision")
    null_scores.append(result["test_score"].mean())
print("AP sous permutation", np.round(null_scores, 3), "prévalence", round(y_dev.mean(), 3))

AP sous permutation [0.118 0.125 0.101 0.122 0.107] prévalence 0.11


## Exercice 3 (20 min). Rédiger pourquoi une CV aléatoire ne prouve pas la performance sur de futurs appels. Proposer une validation temporelle sur le fichier complet ordonné.

In [5]:
print("Le sous-échantillon fourni est aléatoire et n'a pas de date complète. Utiliser le fichier complet ordonné et des blocs temporels, puis auditer les contacts répétés.")

Le sous-échantillon fourni est aléatoire et n'a pas de date complète. Utiliser le fichier complet ordonné et des blocs temporels, puis auditer les contacts répétés.
